# LLM Document Analysis: New Product Releases in SEC 8-K Filings

**CAP 5619 – AI in Fintech, Assignment 2**

This notebook pulls SEC Form 8-K filings for S&P 500 companies from EDGAR, uses a large language model
(Anthropic **Claude Sonnet 5.5** through the API) to run Named Entity Recognition on each filing and pull out
newly announced products, and saves the result as a CSV with the columns:

`company_name, stock_name, filing_time, new_product, product_description`

**Pipeline**

1. **Collect**: S&P 500 tickers → SEC CIK numbers → each company's 8-K list (EDGAR Atom feed).
2. **Preprocess**: download each filing's press-release exhibit (EX-99.x), convert HTML → plain text, keep only
   filings whose text mentions a product launch.
3. **Extract (LLM + NER)**: the LLM tags ORGANIZATION and PRODUCT entities and returns the newly announced
   products as structured JSON.
4. **Output**: clean the results (≤180-char descriptions, deduplicate) and write `output/new_products.csv`.

**Setup:** put `ANTHROPIC_API_KEY=...` and `SEC_USER_AGENT="Your Name your@email"` in a `.env` file
(SEC requires a contact in the User-Agent). Install dependencies with `pip install -r requirements.txt`.

In [ ]:
import csv
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import StringIO
from pathlib import Path

import anthropic
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()  # reads ANTHROPIC_API_KEY and SEC_USER_AGENT from .env

In [ ]:
# ---------------- Configuration ----------------
SEC_HEADERS = {"User-Agent": os.environ["SEC_USER_AGENT"]}  # SEC requires a declared contact
SEC_DELAY = 0.15            # seconds between SEC requests (SEC limit is 10 requests/second)

LLM_MODEL = "claude-sonnet-5-5"   # Anthropic Claude Sonnet 5.5
LLM_EFFORT = "low"                # reasoning effort: extraction is a simple task
LLM_WORKERS = 6                   # parallel LLM requests

FILINGS_PER_COMPANY = 40          # most recent 8-Ks requested per company
MIN_FILING_DATE = "2024-01-01"    # ignore older filings
MAX_CANDIDATE_DOCS = 300          # stop collecting after this many product-related documents
MAX_DOC_CHARS = 20_000            # press releases lead with the announcement; cap length to control cost
MAX_DESC_LEN = 180                # descriptions must be shorter than 180 characters

# 8-K items that can carry product news: 2.02 results, 7.01 Reg FD disclosure, 8.01 other events
RELEVANT_ITEMS = ("2.02", "7.01", "8.01")

# Sectors where product launches are common (banks, utilities, REITs, etc. rarely launch products)
PRODUCT_SECTORS = ["Information Technology", "Health Care", "Communication Services",
                   "Consumer Discretionary", "Consumer Staples", "Industrials"]

CACHE_DIR = Path("cache")         # raw downloads + LLM answers, so re-runs are fast and free
OUTPUT_DIR = Path("output")
for d in (CACHE_DIR / "docs", CACHE_DIR / "llm", OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)